# PROSIT 1 : Petit détour

**Groupe :**
* LUU Philippe - Secrétaire
* AFANE Youcef - Gestionnaire du temps
* RABATEL Antonin - Animateur
* RIVET Alexandre - Scribe

---

## 1. Contexte
Dans le cadre de notre préparation à une mission pour l'ADEME, notre structure CesiCDP nous affecte temporairement à un projet pour la région Grand Est, dirigé par Agathe. L'objectif est d'optimiser les tournées des équipes techniques chargées d'installer des dispositifs connectés sur les lampadaires de la ville. 
Actuellement, les itinéraires sont réalisés « à l'expérience ». Le but est de créer un algorithme capable de calculer le meilleur itinéraire possible : une tournée qui passe une et une seule fois par chaque rue (pour équiper les lampadaires) et revient à son point de départ, minimisant ainsi le temps et le carburant.
Cependant, Agathe soulève plusieurs inquiétudes : ce parcours idéal est-il toujours possible mathématiquement (en référence au problème des ponts de Königsberg) ? Comment gérer les cas où ce n'est pas possible ? Enfin, comment garantir que l'algorithme restera performant (en temps de calcul) sur des villes entières et pas seulement sur de petits quartiers ?

## 2. Mots inconnus / Notions à maîtriser
* **Théorie des graphes :** Modélisation mathématique d'un réseau (ici, les intersections sont des *sommets* et les rues sont des *arêtes*).
* **Graphe Eulérien / Cycle Eulérien :** Un chemin qui passe par *toutes les arêtes* d'un graphe une et une seule fois, et qui revient à son point de départ.
* **Problème des 7 ponts de Königsberg :** Problème mathématique historique résolu par Leonhard Euler, prouvant l'impossibilité de traverser 7 ponts spécifiques une seule fois.
* **Théorème d'Euler :** Conditions nécessaires et suffisantes pour qu'un graphe admette un cycle eulérien (tous les sommets doivent être de degré pair).
* **Arbre Couvrant Minimal (Minimum Spanning Tree) :** Sous-ensemble d'arêtes connectant tous les sommets sans former de cycle, avec le poids total minimum.
* **Complexité algorithmique / asymptotique :** Évaluation théorique (notée O) de la performance d'un algorithme en fonction de la taille des données en entrée (nombre de sommets V et d'arêtes E), indépendamment de la puissance du processeur (CPU).
* **Pire cas (Worst-case) :** La configuration de données qui demandera le plus de temps ou de ressources à l'algorithme pour s'exécuter.

## 3. Problématique
**Comment modéliser un réseau routier urbain sous forme de graphe pour concevoir un algorithme capable de trouver une tournée optimale (passant par toutes les rues), tout en prouvant mathématiquement sa faisabilité et en anticipant sa complexité théorique à grande échelle ?**

## 4. Plan d’action
1. **Modélisation et diagnostic mathématique :** Comprendre la différence entre sommets et arêtes dans le contexte de notre problème (rues vs intersections) et analyser le problème des ponts de Königsberg.
2. **Recherche de la solution optimale (Théorème d'Euler) :** Étudier les cycles eulériens et déterminer ce que l'algorithme doit faire si un tel cycle est impossible (Problème du postier chinois).
3. **Analyse de la pertinence des outils :** Évaluer si l'Arbre Couvrant Minimal est une bonne piste ou une fausse bonne idée pour ce problème précis.
4. **Étude de la complexité algorithmique :** Différencier le temps d'exécution brut (millisecondes CPU) de la complexité asymptotique théorique (O).
5. **Synthèse et préparation du programme :** Poser les bases logiques de l'algorithme à coder dans Jupyter pour rassurer Agathe.

## 5. Réalisation

### 5.1 Modélisation et diagnostic mathématique

### Le problème des ponts de Königsberg

Le problème posé par Agathe rappelle directement le célèbre problème des ponts de Königsberg,
résolu par Euler en 1736. Dans ce problème, on cherche à traverser chaque pont une et une seule
fois en revenant au point de départ.

La clé est de bien identifier ce que représentent les **sommets** et les **arêtes** du graphe :

| Élément du graphe | Problème de Königsberg | Notre problème urbain |
|---|---|---|
| Sommet (nœud) | Île / rive | Intersection |
| Arête | Pont | Rue |

Ce qu'on cherche est un **cycle eulérien** : un chemin qui passe exactement une fois par
chaque *rue* (arête), en revenant au point de départ.

In [1]:
# Liste d'adjacence de la Zone A
liste_zone_A = {
    0: [1, 3],
    1: [0, 4],
    2: [3, 7],
    3: [0, 2, 4, 8],
    4: [1, 3, 5, 7],
    5: [4, 10],
    6: [7, 9],
    7: [2, 4, 6, 8],
    8: [3, 7],
    9: [6, 10],
    10: [5, 9],
}

# Matrice d'adjacence de la Zone A
matrix_zone_A = [
    [0,1,0,1,0,0,0,0,0,0,0],
    [1,0,0,0,1,0,0,0,0,0,0],
    [0,0,0,1,0,0,0,1,0,0,0],
    [1,0,1,0,1,0,0,0,1,0,0],
    [0,1,0,1,0,1,0,1,0,0,0],
    [0,0,0,0,1,0,0,0,0,0,1],
    [0,0,0,0,0,0,0,1,0,1,0],
    [0,0,1,0,1,0,1,0,1,0,0],
    [0,0,0,1,0,0,0,1,0,0,0],
    [0,0,0,0,0,0,1,0,0,0,1],
    [0,0,0,0,0,1,0,0,0,1,0]
]

# Fonctions utilitaires — liste d'adjacence
def degreSommetGraphe(liste, sommet):
    return len(liste[sommet])

def voisinsSommetGraphe(liste, sommet):
    return liste[sommet]

# Fonctions utilitaires — matrice d'adjacence
def voisinsSommetGrapheMatrice(matrice, sommet):
    return [i for i, value in enumerate(matrice[sommet]) if value > 0]

def degreSommetGrapheMatrice(matrice, sommet):
    return sum(matrice[sommet])

# Affichage du degré de chaque sommet
print("Degrés des sommets (Zone A) :")
for sommet in range(len(matrix_zone_A)):
    print(f"  Sommet {sommet} : degré {degreSommetGrapheMatrice(matrix_zone_A, sommet)}")

Degrés des sommets (Zone A) :
  Sommet 0 : degré 2
  Sommet 1 : degré 2
  Sommet 2 : degré 2
  Sommet 3 : degré 4
  Sommet 4 : degré 4
  Sommet 5 : degré 2
  Sommet 6 : degré 2
  Sommet 7 : degré 4
  Sommet 8 : degré 2
  Sommet 9 : degré 2
  Sommet 10 : degré 2


### 5.2. Recherche de la solution optimale — Théorème d'Euler

### Condition d'existence d'un cycle eulérien

Euler a démontré qu'un cycle eulérien existe dans un graphe **si et seulement si
tous les sommets sont de degré pair**.

Le degré d'un sommet est le nombre d'arêtes qui y sont connectées, c'est-à-dire
le nombre de rues qui partent de cette intersection.

**Intuition** : à chaque fois qu'on entre dans une intersection, il faut pouvoir en ressortir
par une rue non encore empruntée. Si un sommet a un degré impair, on finira
inévitablement bloqué.

### Que faire si le cycle eulérien est impossible ?

Si certains sommets ont un degré impair, on ne peut pas éviter de repasser par certaines rues.
Le **problème du postier chinois** (*Chinese Postman Problem*) résout ce cas :
on cherche le circuit de longueur minimale qui couvre toutes les arêtes,
en autorisant de repasser sur certaines rues le moins possible.

La stratégie est la suivante :
1. Identifier les sommets de degré impair
2. Les appairer de façon optimale (coût de trajet minimal entre chaque paire)
3. Dupliquer les arêtes des chemins reliant ces paires → le graphe devient eulérien
4. Calculer le cycle eulérien sur ce graphe augmenté

In [2]:
import copy
from collections import deque

# Vérification de l'existence d'un cycle eulérien
def existeCycleEulerien(matrice):
    for sommet in range(len(matrice)):
        if degreSommetGrapheMatrice(matrice, sommet) % 2 != 0:
            return False
    return True

# Identification des sommets de degré impair
def sommetsDegreImpair(matrice):
    return [s for s in range(len(matrice))
            if degreSommetGrapheMatrice(matrice, s) % 2 != 0]

# Algorithme de Hierholzer — calcul du cycle eulérien
def cycleEulerien(matrice, depart=0):
    if not existeCycleEulerien(matrice):
        return None
    matrice = copy.deepcopy(matrice)
    n = len(matrice)
    cycle = deque()
    stack = deque()
    cur = depart
    while len(stack) > 0 or degreSommetGrapheMatrice(matrice, cur) != 0:
        if degreSommetGrapheMatrice(matrice, cur) == 0:
            cycle.appendleft(cur)
            cur = stack.pop()
        else:
            for i in range(n):
                if matrice[cur][i] > 0:
                    stack.append(cur)
                    matrice[cur][i] -= 1
                    matrice[i][cur] -= 1
                    cur = i
                    break
    cycle.appendleft(cur)
    return list(cycle)

# Diagnostic et résultat sur la Zone A
impairs = sommetsDegreImpair(matrix_zone_A)
print(f"Sommets de degré impair : {impairs}")

if existeCycleEulerien(matrix_zone_A):
    print("Un cycle eulérien existe.")
    cycle = cycleEulerien(matrix_zone_A, depart=0)
    print(f"Cycle eulérien : {cycle}")
else:
    print("Pas de cycle eulérien possible.")
    print("→ Il faudra appliquer le problème du postier chinois.")

Sommets de degré impair : []
Un cycle eulérien existe.
Cycle eulérien : [0, 1, 4, 3, 2, 7, 4, 5, 10, 9, 6, 7, 8, 3, 0]


### 5.3. L'Arbre Couvrant Minimal : bonne piste ou fausse bonne idée ?

### Qu'est-ce que l'Arbre Couvrant Minimal (ACM) ?

L'ACM (ou *Minimum Spanning Tree*) est un sous-graphe qui connecte tous les sommets
d'un graphe avec le coût total minimum, sans former de cycle.
Les algorithmes classiques sont **Kruskal** et **Prim**.

### Pourquoi ça ne répond pas à notre problème

| Critère | ACM | Cycle Eulérien |
|---|---|---|
| Objectif | Relier tous les sommets | Parcourir toutes les arêtes |
| Présence de cycles | Aucun cycle | Un seul grand cycle |
| Arêtes visitées | Sous-ensemble des arêtes | Toutes les arêtes |
| Usage typique | Réseau câblé, pipelines | Tournées de maintenance |

L'ACM cherche à *éviter* les cycles, alors que notre problème consiste précisément
à *construire* un cycle. De plus, l'ACM ne garantit pas de passer par toutes les rues —
il peut en omettre pour minimiser le coût total.

**Conclusion** : l'ACM est une fausse bonne idée pour ce problème.
L'algorithme adapté est **Hierholzer**, implémenté à la section précédente.

### 5.4. Complexité algorithmique

### Temps d'exécution brut vs complexité asymptotique

Le temps mesuré en millisecondes dépend de la machine, de la charge CPU au moment
de l'exécution, et du langage. Il donne une **indication pratique** mais ne permet
pas de prédire le comportement sur de grandes instances.

La **complexité asymptotique** $\mathcal{O}$ mesure comment le nombre d'opérations
évolue en fonction de la taille de l'entrée, indépendamment du matériel.

### Complexité de nos algorithmes

| Algorithme | Complexité (matrice d'adjacence) | Variable dominante |
|---|---|---|
| `existeCycleEulerien` | $\mathcal{O}(V^2)$ | Nombre de sommets $V$ |
| `cycleEulerien` (Hierholzer) | $\mathcal{O}(E\cdot V)$ (jusqu'à $\mathcal{O}(V^3)$ si $E \in \Theta(V^2)$) | $V$ et $E$ |
| `sommetsDegreImpair` | $\mathcal{O}(V)$ | Nombre de sommets $V$ |

Avec notre **représentation en matrice d'adjacence**, le calcul du degré d'un sommet
et la recherche de la prochaine arête disponible nécessitent de balayer une ligne de
la matrice, ce qui coûte $\mathcal{O}(V)$ par arête. L'algorithme de Hierholzer visite
bien chaque arête **exactement une fois**, mais le coût total devient donc
$\mathcal{O}(E \cdot V)$ (et jusqu'à $\mathcal{O}(V^3)$ sur un graphe dense où
$E \in \Theta(V^2)$).

Avec une **liste d'adjacence** et/ou des degrés pré‑calculés, la même logique
donnerait une complexité $\mathcal{O}(V + E)$ pour `existeCycleEulerien` et
$\mathcal{O}(E)$ pour `cycleEulerien`.

### Cas limites

- **Rue centrale unique** (graphe linéaire) : très peu d'arêtes, $E \approx V$.
  L'algorithme reste raisonnablement rapide, même avec la matrice.
- **Ville dense** (graphe complet) : $E \approx V^2$. Avec la matrice, la complexité de Hierholzer devient $\mathcal{O}(V^3)$. Si de nombreux sommets sont de degré impair,
  pour Hierholzer, mais si de nombreux sommets sont de degré impair,
  la phase d'appariement du postier chinois peut atteindre $\mathcal{O}(V^3)$.
- **Pire cas absolu** : graphe complet avec tous les sommets de degré impair →
  appariement parfait de poids minimum, très coûteux.

In [3]:
import time

def cycleEulerien_chrono(matrice, depart=0):
    debut = time.process_time()
    resultat = cycleEulerien(matrice, depart)
    fin = time.process_time()
    temps_ms = (fin - debut) * 1000
    return resultat, temps_ms

cycle, temps = cycleEulerien_chrono(matrix_zone_A, depart=0)
print(f"Cycle : {cycle}")
print(f"Temps CPU : {temps:.4f} ms")

Cycle : [0, 1, 4, 3, 2, 7, 4, 5, 10, 9, 6, 7, 8, 3, 0]
Temps CPU : 0.1180 ms


## 5.5. Synthèse — Architecture logique du programme

Le programme final suit la logique suivante :

In [4]:
def tourneeOptimale(matrice, depart=0):
    """
    Fonction principale.
    Calcule une tournée couvrant toutes les rues de la zone et retourne un
    tuple `(itineraire, temps_ms)` où :
        - `itineraire` est une liste de sommets représentant l'itinéraire
            optimal lorsque le graphe admet un cycle eulérien ;
        - `itineraire` vaut `None` lorsque le graphe n'est pas eulérien (cas du
            postier chinois non encore implémenté dans cette fonction).
        `temps_ms` est toujours le temps de calcul en millisecondes.
    """
    debut = time.process_time()

    impairs = sommetsDegreImpair(matrice)

    if existeCycleEulerien(matrice):
        print("Cycle eulérien possible — solution exacte.")
        itineraire = cycleEulerien(matrice, depart)
    else:
        print(f"{len(impairs)} sommets de degré impair détectés : {impairs}")
        print("Cycle eulérien impossible — application du postier chinois.")
        raise NotImplementedError(
            "Algorithme du postier chinois non implémenté pour les graphes non eulériens."
        )

    fin = time.process_time()
    temps_ms = (fin - debut) * 1000

    return itineraire, temps_ms

# Exécution sur la Zone A
itineraire, temps = tourneeOptimale(matrix_zone_A, depart=0)
print(f"\nItinéraire : {itineraire}")
print(f"Temps de calcul : {temps:.4f} ms")

Cycle eulérien possible — solution exacte.

Itinéraire : [0, 1, 4, 3, 2, 7, 4, 5, 10, 9, 6, 7, 8, 3, 0]
Temps de calcul : 0.4820 ms
